In [0]:
%pip install psycopg2-binary
dbutils.library.restartPython()
display(spark.sql("SHOW TABLES IN workspace.personal_finance_gold"))

In [0]:

# Databricks notebook source
# MAGIC %md
# MAGIC # Serving: sync gold tables to Postgres (Neon)
# MAGIC Copies gold tables into the Postgres database the TrueNorth agent reads from.
# MAGIC
# MAGIC - **Typed from the Spark schema.** Decimals stay `NUMERIC` and dates stay `DATE` in Postgres, not text.
# MAGIC - **Idempotent and atomic.** Each table loads into a staging table, then swaps in within one transaction. Re-running never duplicates rows, and the agent never reads a half-loaded table.
# MAGIC - **Gold only.** This never touches `members`, `accounts` or `transactions`, so the planted test members (IDs 1-6) stay intact.
# MAGIC - Built for small gold tables (collected to the driver). For large tables, use the Spark JDBC writer instead.
 
# COMMAND ----------
 
import psycopg2
from psycopg2.extras import execute_values
from pyspark.sql.types import (
    BooleanType, DateType, DecimalType, DoubleType, FloatType,
    IntegerType, LongType, ShortType, StringType, TimestampType,
)
 
NEON_HOST = "ep-rapid-recipe-b5qgla8w-pooler.c-7.us-east-2.aws.neon.tech"
NEON_DB = "neondb"
NEON_USER = "neondb_owner"
NEON_PASSWORD = dbutils.secrets.get(scope="neon-scope", key="neon-password")
 
CATALOG = "workspace"
GOLD_SCHEMA = "personal_finance_gold"
GOLD_TABLES = ["dim_tfsa_limits", "fct_boc_policy_rates"]
 
# COMMAND ----------
 
def pg_type(spark_type):
    """Map a Spark column type to its Postgres equivalent."""
    if isinstance(spark_type, DecimalType):
        return f"NUMERIC({spark_type.precision}, {spark_type.scale})"
    mapping = {
        IntegerType: "INTEGER",
        ShortType: "SMALLINT",
        LongType: "BIGINT",
        DoubleType: "DOUBLE PRECISION",
        FloatType: "REAL",
        BooleanType: "BOOLEAN",
        DateType: "DATE",
        TimestampType: "TIMESTAMPTZ",
        StringType: "TEXT",
    }
    return mapping.get(type(spark_type), "TEXT")
 
 
def sync_gold_table(table_name, target_name=None):
    target = target_name or table_name
    staging = f"{target}__staging"
    source = f"{CATALOG}.{GOLD_SCHEMA}.{table_name}"
 
    df = spark.table(source)
    rows = [tuple(r) for r in df.collect()]   # keeps Decimal and date as native Python types
    if not rows:
        print(f"Skipping {source}: table is empty")
        return
 
    columns_ddl = ", ".join(f'"{f.name}" {pg_type(f.dataType)}' for f in df.schema.fields)
    column_list = ", ".join(f'"{c}"' for c in df.columns)
 
    conn = psycopg2.connect(
        host=NEON_HOST, dbname=NEON_DB, user=NEON_USER,
        password=NEON_PASSWORD, port=5432, sslmode="require",
    )
    try:
        with conn.cursor() as cur:
            # Everything below runs in one transaction: swap or nothing
            cur.execute(f'DROP TABLE IF EXISTS "{staging}"')
            cur.execute(f'CREATE TABLE "{staging}" ({columns_ddl})')
            execute_values(cur, f'INSERT INTO "{staging}" ({column_list}) VALUES %s', rows)
            cur.execute(f'DROP TABLE IF EXISTS "{target}"')
            cur.execute(f'ALTER TABLE "{staging}" RENAME TO "{target}"')
        conn.commit()
        print(f"Synced {len(rows):,} rows: {source} -> {target}")
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()
 
# COMMAND ----------
 
failures = []
for table in GOLD_TABLES:
    try:
        sync_gold_table(table)
    except Exception as e:
        failures.append(table)
        print(f"Failed to sync {table}: {e}")
 
# Fail the job loudly instead of reporting success with errors
assert not failures, f"Sync failed for: {failures}"
print("Gold sync complete.")
 